In [5]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from IPython.display import display

In [6]:
# Data prepration 
dataFrame = pd.read_csv('course_lead_scoring_2026_hw3.csv', delimiter=',')

# substitute NA / 0 for non existent values
numeric_cols = dataFrame.select_dtypes(include="number").columns
string_cols = dataFrame.select_dtypes(include=["object", "string"]).columns
display(numeric_cols)
display("null numeric columns :" + str(dataFrame[numeric_cols].isnull().sum()))
display(string_cols)
display("null string columns :" + str(dataFrame[string_cols].isnull().sum()))
dataFrame[numeric_cols] = dataFrame[numeric_cols].fillna(0)
dataFrame[string_cols] = dataFrame[string_cols].fillna("NA")
display("after fixed null numeric columns :" + str(dataFrame[numeric_cols].isnull().sum()))
display("after fixed null string columns :" + str(dataFrame[string_cols].isnull().sum()))


Index(['annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='str')

'null numeric columns :annual_income               369\nnumber_of_courses_viewed      0\ninteraction_count             0\nlead_score                   35\nconverted                     0\ndtype: int64'

Index(['lead_source', 'industry', 'employment_status', 'location'], dtype='str')

'null string columns :lead_source          148\nindustry             240\nemployment_status    194\nlocation             208\ndtype: int64'

'after fixed null numeric columns :annual_income               0\nnumber_of_courses_viewed    0\ninteraction_count           0\nlead_score                  0\nconverted                   0\ndtype: int64'

'after fixed null string columns :lead_source          0\nindustry             0\nemployment_status    0\nlocation             0\ndtype: int64'

In [7]:
# 1. most frequent observation (mode) for the column industry
industry_mode = dataFrame['industry'].mode()[0]
display("most frequent observation for the column industry: " + str(industry_mode))

# Split the data
df_full_train, df_test = train_test_split(dataFrame, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)


'most frequent observation for the column industry: technology'

In [8]:
#2. Biggest correlation
correlation_matrix = dataFrame[numeric_cols].corr()
display(correlation_matrix)
abs_correlations = correlation_matrix.abs().to_numpy(copy=True)

# Exclude self-correlations.
np.fill_diagonal(abs_correlations, np.nan)

#find the largest correlation index in the frame
max_corr_idx = np.unravel_index(
    np.nanargmax(abs_correlations), abs_correlations.shape
)
max_corr_value = abs_correlations[max_corr_idx]
display("Maximum correlation value: " + str(max_corr_value))

display(
    f"Biggest absolute correlation is between "
    f"{correlation_matrix.columns[max_corr_idx[0]]} and "
    f"{correlation_matrix.columns[max_corr_idx[1]]}: {max_corr_value:.4f}"
)

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


'Maximum correlation value: 0.9157457980418697'

'Biggest absolute correlation is between interaction_count and lead_score: 0.9157'

In [9]:
#3. Biggest MI (Mutual Information)
#Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
#Round the scores to 2 decimals using round(score, 2).
from sklearn.metrics import mutual_info_score

def mutual_info_score_func(series):
    return mutual_info_score(series, df_train["converted"])

categorical = df_train.drop(columns=["converted"]).select_dtypes(
    include=["object", "string", "category"]
).columns

mi_scores = df_train[categorical].apply(mutual_info_score_func)

display(mi_scores.sort_values(ascending=False).apply(lambda score: round(score, 2)))

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [ ]:
#4 .Now let's train a logistic regression.
#Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
# One-hot encode categorical variables
# Align the validation set with the training set
from sklearn.feature_extraction import DictVectorizer

dicts = df_train.drop(columns=['converted']).to_dict(orient='records')
dv = DictVectorizer(sparse=False)
#	Discover all categories and encode the training data
X_train_encoded = dv.fit_transform(dicts)
display(dv.get_feature_names_out())
display(X_train_encoded)

dicts_val = df_val.drop(columns=['converted']).to_dict(orient='records')
#Encode using only the categories already discovered from training
X_valid_encoded = dv.transform(dicts_val)
display(dv.get_feature_names_out())
display(X_valid_encoded)

# Calculate the accuracy on the validation dataset and round it to 2 decimal digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Fit the model on the training dataset
#To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train_encoded, df_train['converted'])
y_pred = model.predict(X_valid_encoded)
accuracy = accuracy_score(df_val['converted'], y_pred)
accuracy = round(accuracy, 2)
accuracy



array(['annual_income', 'employment_status=NA',
       'employment_status=employed', 'employment_status=self_employed',
       'employment_status=student', 'employment_status=unemployed',
       'industry=NA', 'industry=education', 'industry=finance',
       'industry=healthcare', 'industry=manufacturing', 'industry=retail',
       'industry=technology', 'interaction_count', 'lead_score',
       'lead_source=NA', 'lead_source=events',
       'lead_source=organic_search', 'lead_source=paid_ads',
       'lead_source=referral', 'lead_source=social_media', 'location=NA',
       'location=africa', 'location=asia', 'location=europe',
       'location=north_america', 'location=south_america',
       'number_of_courses_viewed'], dtype=object)

array([[6.5980e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        2.0000e+00],
       [7.1662e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 1.0000e+00,
        0.0000e+00],
       [4.0220e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        2.0000e+00],
       ...,
       [7.1532e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        3.0000e+00],
       [4.8836e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00],
       [6.8969e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00]], shape=(3000, 28))

array(['annual_income', 'employment_status=NA',
       'employment_status=employed', 'employment_status=self_employed',
       'employment_status=student', 'employment_status=unemployed',
       'industry=NA', 'industry=education', 'industry=finance',
       'industry=healthcare', 'industry=manufacturing', 'industry=retail',
       'industry=technology', 'interaction_count', 'lead_score',
       'lead_source=NA', 'lead_source=events',
       'lead_source=organic_search', 'lead_source=paid_ads',
       'lead_source=referral', 'lead_source=social_media', 'location=NA',
       'location=africa', 'location=asia', 'location=europe',
       'location=north_america', 'location=south_america',
       'number_of_courses_viewed'], dtype=object)

array([[8.0457e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        3.0000e+00],
       [7.4480e+04, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00],
       [4.3944e+04, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        2.0000e+00],
       ...,
       [6.2439e+04, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00],
       [4.0878e+04, 0.0000e+00, 0.0000e+00, ..., 1.0000e+00, 0.0000e+00,
        1.0000e+00],
       [4.8769e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        3.0000e+00]], shape=(1000, 28))

0.65

In [34]:
#5 Feature selection
'''Let's find the least useful feature using the feature elimination technique.
Train a model using the same features and parameters as in Q4 (without rounding).
Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
For each feature, calculate the difference between the original accuracy and the accuracy without the feature.
Which of following feature has the smallest difference?'''

# Original model & accuracy (without rounding)
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train_encoded, df_train['converted'])
y_pred = model.predict(X_valid_encoded)
original_accuracy = accuracy_score(df_val['converted'], y_pred)
print(f"Original accuracy: {original_accuracy:.6f}\n")

# All features in the dataset (excluding the target)
features = [col for col in df_train.columns if col != 'converted']

feature_differences = {}

for feature in features:
    # Exclude this one feature
    subset = [col for col in features if col != feature]
    
    # Re-vectorize without this feature
    dv_temp = DictVectorizer(sparse=False)
    X_train_temp = dv_temp.fit_transform(df_train[subset].to_dict(orient='records'))
    X_valid_temp = dv_temp.transform(df_val[subset].to_dict(orient='records'))
    
    # Train model without this feature
    model_temp = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_temp.fit(X_train_temp, df_train['converted'])
    
    # Evaluate accuracy and compute difference
    y_pred_temp = model_temp.predict(X_valid_temp)
    accuracy_temp = accuracy_score(df_val['converted'], y_pred_temp)
    
    # Difference: original accuracy - accuracy without feature
    diff = original_accuracy - accuracy_temp
    feature_differences[feature] = diff
    print(f"Without {feature:25s}: accuracy = {accuracy_temp:.6f}, difference = {diff:+.6f}")

print("\n--- Summary of differences ---")
display(feature_differences)

Original accuracy: 0.645000

Without lead_source              : accuracy = 0.642000, difference = +0.003000
Without industry                 : accuracy = 0.645000, difference = +0.000000
Without employment_status        : accuracy = 0.645000, difference = +0.000000
Without location                 : accuracy = 0.645000, difference = +0.000000
Without annual_income            : accuracy = 0.724000, difference = -0.079000
Without number_of_courses_viewed : accuracy = 0.643000, difference = +0.002000
Without interaction_count        : accuracy = 0.601000, difference = +0.044000
Without lead_score               : accuracy = 0.644000, difference = +0.001000

--- Summary of differences ---


{'lead_source': 0.0030000000000000027,
 'industry': 0.0,
 'employment_status': 0.0,
 'location': 0.0,
 'annual_income': -0.07899999999999996,
 'number_of_courses_viewed': 0.0020000000000000018,
 'interaction_count': 0.04400000000000004,
 'lead_score': 0.0010000000000000009}

In [35]:
#6 6. Parameter tuning 
'''6
Now let's train a regularized logistic regression.
Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
Train models using all the features as in Q4.
Calculate the accuracy on the validation dataset and round it to 3 decimal digits.'''
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train_encoded, df_train['converted'])
    y_pred = model.predict(X_valid_encoded)
    accuracy = accuracy_score(df_val['converted'], y_pred)
    accuracy = round(accuracy, 3)
    print(f"C={C}, Accuracy={accuracy}")

C=1e-06, Accuracy=0.598
C=1e-05, Accuracy=0.598
C=0.0001, Accuracy=0.613
C=0.001, Accuracy=0.645
